# 🌌 OpenVista - Google Colab Edition

This notebook allows you to run the full OpenVista AI platform on Google Colab, leveraging a cloud GPU for image generation.

### 🛠️ Step 1: Setup Environment
Clone the repository and install all dependencies.

In [ ]:
# @title Clone and Bootstrap
import os
import sys
if not os.path.exists("OpenVista"):
    !git clone https://github.com/FayssalSabri/OpenVista.git

%cd /content/OpenVista

!bash scripts/colab_bootstrap.sh

# Force install critical modules in the current kernel
!{sys.executable} -m pip install -q -U celery fastapi uvicorn pyngrok nest-asyncio python-dotenv redis diffusers transformers accelerate

### 🔑 Step 2: Configure Ngrok
To access the web interface reliably, we use Ngrok for the Frontend (which now proxies the backend automatically).
1. Get your token here: [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)
2. Paste it below.

In [ ]:
# @title Ngrok Auth
import os
NGROK_TOKEN = "" # @param {type:"string"}
os.environ["NGROK_TOKEN"] = NGROK_TOKEN

from pyngrok import ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    !pkill ngrok
else:
    print("❌ Please provide an NGROK_TOKEN!")

### 🚀 Step 3: Start Services
This will start the Backend (FastAPI), the AI Worker (Celery), and the Frontend (Next.js).

In [ ]:
# @title Run OpenVista
import subprocess
import time
import sys
import os
import threading
import requests
from pyngrok import ngrok
from google.colab import output

def log_reader(pipe, prefix):
    for line in iter(pipe.readline, b''):
        print(f"{prefix}: {line.decode().strip()}")

# 1. Setup Tunnels
print("📡 Setting up Tunnel (Frontend on 3000)...")
try:
    ngrok.kill()
    # We tunnel the frontend. The frontend is configured to proxy /api to the backend on 8000.
    frontend_tunnel = ngrok.connect(3000, name="openvista_frontend")
    frontend_url = frontend_tunnel.public_url
except Exception as e:
    print(f"⚠️ Ngrok failed ({e}), falling back to Colab Proxy...")
    raw_frontend_url = output.eval_js('google.colab.kernel.proxyPort(3000)')
    frontend_url = raw_frontend_url.strip("/")
    if not frontend_url.startswith("http"):
        frontend_url = f"https://{frontend_url}"

print(f"\n🔥 PUBLIC URL: {frontend_url}\n")

# 2. Update Environment Variables
# Write to .env, backend/.env and frontend/.env.local
env_content = f"""REDIS_URL=redis://localhost:6379/0
NEXT_PUBLIC_API_URL=/api
DATA_PATH=/content/OpenVista/data
MOCK_WORKER=false
"""
with open(".env", "w") as f: f.write(env_content)
with open("backend/.env", "w") as f: f.write(env_content)
os.makedirs("frontend", exist_ok=True)
with open("frontend/.env.local", "w") as f: f.write(env_content)

# 3. Prepare Environment
custom_env = os.environ.copy()
backend_path = os.path.join(os.getcwd(), "backend")
custom_env["PYTHONPATH"] = f"{backend_path}:{custom_env.get('PYTHONPATH', '')}"

# 4. Start Backend
print("🟢 Starting Backend (Port 8000)...")
backend_proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"], 
                               cwd="backend", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=custom_env)
threading.Thread(target=log_reader, args=(backend_proc.stdout, "Backend"), daemon=True).start()

# 5. Wait for Backend
print("⏳ Waiting for Backend to initialize...")
for i in range(30):
    try:
        res = requests.get("http://localhost:8000/")
        if res.status_code == 200:
            print("✅ Backend is UP!")
            break
    except:
        pass
    time.sleep(2)

# 6. Start Worker
print("🟢 Starting Worker...")
worker_proc = subprocess.Popen([sys.executable, "-m", "celery", "-A", "worker", "worker", "--loglevel=info"], 
                               cwd="backend", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=custom_env)
threading.Thread(target=log_reader, args=(worker_proc.stdout, "Worker"), daemon=True).start()

# 7. Start Frontend
print("🟢 Starting Frontend (Port 3000)...")
frontend_proc = subprocess.Popen(["npm", "run", "dev", "--", "-p", "3000", "-H", "0.0.0.0"], 
                                 cwd="frontend", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=custom_env)
threading.Thread(target=log_reader, args=(frontend_proc.stdout, "Frontend"), daemon=True).start()

print(f"\n✨ Access OpenVista at: {frontend_url}")
print("The frontend will automatically proxy /api requests to the local backend.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Stopping services...")
    backend_proc.terminate()
    worker_proc.terminate()
    frontend_proc.terminate()
    ngrok.kill()